# Cohort Selection for Longitudinal ECG Analysis

This notebook identifies patients from the Harvard-Emory-ECG Database (HEEDB I0006) subcohort who had no heart failure (HF) diagnosis at baseline (at the time of ECG) and separates them into patients who subsequently developed HF and those who remained HF-free during follow-up. The resulting cohort and relevant dates are prepared for subsequent time-to-event analysis of longitudinal ECG biomarkers.

#### To download the HEEDB data please visit: 
https://bdsp.io/content/heedb/5.0/

#### Please cite the original database paper when using this data: 

Koscova, Z., Li, Q., Robichaux, C. et al. The Harvard-Emory ECG Database. Sci Data 13, 516 (2026). https://doi.org/10.1038/s41597-026-06861-9


In [ ]:
import numpy as np
import pandas as pd
import os

In [ ]:
# Read the metadata diagnoses file and diagnoses dictionary file
##### Please provide path to the the location of HEEDB database and I0006 subfolder on your device 
path = ''
metadata  = pd.read_csv(os.path.join(path,'metadata/metadata.csv'))
diagnoses_dictionary = pd.read_csv(os.path.join(path, '12SL_diagnoses/diagnoses_dictionary.csv'))
diagnoses = pd.read_csv(os.path.join(path,'12SL_diagnoses/diagnoses.csv'))

In [ ]:
# Convert the string to a list of integers
diagnoses = diagnoses[~pd.isna(diagnoses['codes'])]
diagnoses['codes'] = diagnoses['codes'].apply(lambda x: [int(i.strip()) for i in x.split(',')])

## These are 12SL codes for technical problems with the ECGs, we want to filter those to exclude the ECG with incomplete lengths, lead swaps or being all zeros
technical_problems = [1500, 1501, 1502, 1503, 1504, 1505, 1672, 1673, 1676, 1678, 1679] # noise, not enough qrs complexes for analysis and so on
# Keep only rows where no code is in technical_problems
diagnoses = diagnoses[~diagnoses['codes'].apply(lambda lst: any(code in technical_problems for code in lst))]


In [ ]:
# Merge metadata with diagnoses, delete patients that don't have BDSPPatientID
metadata = metadata[~pd.isna(metadata['BDSPPatientID'])]
metadata_diagnoses = pd.merge(metadata, diagnoses, on = 'FileName', how = 'inner')
metadata_diagnoses = metadata_diagnoses[~pd.isna(metadata_diagnoses['BDSPPatientID'])]

In [ ]:
### Extract icd10 hf codes
icd10 = pd.read_csv(os.path.join(path,'ICD_codes/icd10_codes.csv'))
hf_pattern = r'^(?:I50(?:\.\d*)?|428(?:\.\d*)?)'
df_hf = icd10[icd10['DIAGNOSIS_ICD10_CD'].str.contains(hf_pattern, case=False, na=False)]
df_hf['RECORDED_DT'] = pd.to_datetime(df_hf['RECORDED_DT'])
df_hf = df_hf.groupby('BDSPPatientID')['RECORDED_DT'].min().reset_index()

In [ ]:

### Extract icd9 hf codes
icd9 = pd.read_csv(os.path.join(path,'ICD_codes/icd9_codes.csv'))
hf_pattern = r'^(?:I50(?:\.\d*)?|428(?:\.\d*)?)'
df_hf2 = icd9[icd9['DIAGNOSIS_ICD9_CD'].str.contains(hf_pattern, case=False, na=False)]
df_hf2['RECORDED_DT'] = pd.to_datetime(df_hf2['RECORDED_DT'])
df_hf2 = df_hf2.groupby('BDSPPatientID')['RECORDED_DT'].min().reset_index()

In [ ]:
# Concat icd10 and icd9 codes
df_hf_final = pd.concat([df_hf, df_hf2], axis = 0, ignore_index = True)
# Select the single hf code for each patient with minimum date of diagnoses
df_hf_final = df_hf_final.groupby('BDSPPatientID')['RECORDED_DT'].min().reset_index()
print(df_hf_final.shape)

In [ ]:
# Store the CSV with patient ID and HF diagnosis date
#df_hf_final.to_csv('HF_codes_HEEDB.csv', index = False)

In [ ]:
# Store last visit date of patients that do not have HF recorded

codes_all = pd.concat([icd9[['BDSPPatientID', 'RECORDED_DT']], icd10[['BDSPPatientID', 'RECORDED_DT']]],axis = 0, ignore_index = True)
patients_final = codes_all.groupby('BDSPPatientID')['RECORDED_DT'].max().reset_index()

#patients_final.to_csv('last_visit_date_HEEDB.csv', index = False)